In [5]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("src exists?", (ROOT / "src").exists())

ROOT: /data/ephemeral/pro-nlp-finalproject-nlp-13
src exists? True


In [6]:
import pandas as pd
import os
from openai import OpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv

from src.freshqa.messages import build_messages
from src.freshqa.utils import safe_parse_json, get_today
from src.freshqa.llm import call_solar_pro2, LLMConfig
from src.freshqa.router import route_question
from src.freshqa.retrieve import WikipediaRetriever, WikiConfig
from src.freshqa.keyword import extract_keyword

In [7]:
df = pd.read_csv("/data/ephemeral/pro-nlp-finalproject-nlp-13/data/ko-freshqa_2025_dev.csv", index_col=False) 
TODAY = get_today() 
TODAY

'2026-02-03'

In [8]:
SYSTEM_PROMPT_ROUTER = """
당신은 신중한 지식 어시스턴트입니다.
질문에 답할 때, 확실히 아는 것만 답하고 불확실하면 솔직히 말하세요.

### [기준 정보]
- TODAY: {TODAY}
- 지식 컷오프: 2025년 1월 말

### [출력 형식: 반드시 준수]
- 반드시 아래 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)
- 키는 정확히 3개만 사용:
{{
  "answer_type": "A/B/C/D",
  "answer": "최종 답변(짧고 직접적)",
  "reasoning": "판단 근거"
}}

### [최우선 원칙]
1) **환각 금지**: 모르는 고유명사/숫자/날짜/기관명을 만들지 마세요.
2) **불필요 확장 금지**: 질문이 요구하지 않은 디테일(작품명, 제조사, 법률번호, 세부 수치, 추가 인물 등)을 덧붙이지 마세요.
3) **최소 답변 원칙**: 가능한 한 **정답만** 말하세요. (한 문장 선호)
4) **전제 점검 우선**: 질문 자체의 전제가 성립하는지 먼저 판단하세요.

### [Type 결정 규칙]

#### Type A (확실한 지식 / 계산 가능)
- 아래에 해당할 때만 A로 답하세요.
  - 변하지 않는 역사적 사실/기본 상식/정의/용어(예: 무어의 법칙, 산업혁명 시작 나라)
  - TODAY와 무관한 고정 정보
  - **계산 가능**: 생년/날짜가 질문에 주어졌거나 널리 확정된 경우의 산술(나이/기간/단순 수학)
- A일 때도 **질문이 묻는 범위까지만** 답하세요. (예: “언제?”면 연도만)

#### Type B (확인 필요: 최신/변동/정밀 데이터)
- 아래 중 하나라도 해당하면 기본적으로 B입니다(검색 없이는 단정 금지).
  - “현재/가장 최근/최신/올해/이번년도/마지막/역대/최고/1위” 등 최신성 트리거
  - 정책/법/제도/지원조건/연령·소득·자산 기준/기관 조직 신설·명칭
  - 스포츠 결과, 주가/환율/통계 수치(기준일에 따라 달라지는 값), 기업 실적
  - 촬영지/로케이션/세부 사건 일자처럼 자료 확인이 필요한 사실
  - “총 몇 개/몇 명/몇 마리/몇 개국”처럼 카운트가 최신 정보나 정확한 목록에 의존하는 질문
- B의 answer는 아래 중 하나로 **짧게** 답하세요:
  - "웹 검색이 필요합니다."
  - "확인되지 않았습니다."
  - "발표된 바 없습니다."
  (상황에 맞는 것 하나만 선택)

#### Type C (전제 오류)
- 질문 전제가 성립하지 않으면 C입니다.
  - 존재하지 않는 대상/사건, 시간적 모순, 잘못된 관계·소속, 사실 왜곡 등
- **C의 answer는 ‘교정된 사실 1문장’만** 쓰세요.
  - 금지: "전제가 틀렸습니다", "질문이 잘못되었습니다" 같은 메타 문구
  - 허용: "제주도는 조선의 수도였던 적이 없습니다."

#### Type D (미래 예측)
- 미래의 불확실한 예측을 요구하면 D입니다.
- answer: "예측할 수 없습니다."

### [reasoning 작성 규칙]
- reasoning은 **1문장**으로, 왜 A/B/C/D인지 근거만 말하세요.
- answer를 반복하거나 길게 설명하지 마세요.

### [예시]
{{
  "answer_type": "A",
  "answer": "무어의 법칙입니다.",
  "reasoning": "반도체 집적도 증가를 설명하는 널리 확립된 용어입니다."
}}
{{
  "answer_type": "B",
  "answer": "웹 검색이 필요합니다.",
  "reasoning": "최신/변동 가능 정보라 컷오프 이후 확인이 필요합니다."
}}
{{
  "answer_type": "C",
  "answer": "대한민국에는 국왕이 존재하지 않습니다.",
  "reasoning": "질문의 전제가 헌정 체제와 맞지 않습니다."
}}
"""

USER_PROMPT_ROUTER = """
### [입력]
QUESTION: {QUESTION}
"""

In [9]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)

cfg = LLMConfig(
    model="solar-pro2",
    temperature=0.0,
    top_p=1.0,
    reasoning_effort="high"
)

def llm_call(messages):
    return call_solar_pro2(client=client, messages=messages, cfg=cfg)

In [10]:
q = df.loc[0, "question"]

raw, route = route_question(
    question=q,
    today=TODAY,
    system_prompt=SYSTEM_PROMPT_ROUTER,
    user_prompt=USER_PROMPT_ROUTER,
    llm_call=llm_call,
)

raw, route

('{\n  "answer_type": "B",\n  "answer": "웹 검색이 필요합니다.",\n  "reasoning": "APEC 정상회의 횟수는 최신 정보에 따라 변동 가능하며, 컷오프 이후 개최된 회의 정보가 필요합니다."\n}',
 RouteResult(answer_type='B', answer='웹 검색이 필요합니다.', reasoning='APEC 정상회의 횟수는 최신 정보에 따라 변동 가능하며, 컷오프 이후 개최된 회의 정보가 필요합니다.'))

In [11]:
import json
obj = json.loads(raw)
obj.keys(), obj["answer_type"], obj["answer"]

(dict_keys(['answer_type', 'answer', 'reasoning']), 'B', '웹 검색이 필요합니다.')

In [12]:
SYSTEM_PROMPT_KEYWORD = """
당신은 위키피디아 검색용 키워드 추출 전문가입니다. 
사용자의 질문을 분석하여 위키피디아 검색에 가장 적합한 '문서 제목'을 결정하고, 반드시 아래 JSON 형식으로만 출력하세요.

### 규칙:
1. 반드시 순수 JSON만 출력하세요. (마크다운, 사족 금지)
2. `query` 키에 위키피디아 문서 제목을 담으세요.
3. 인물은 직업을 괄호 안에 명시합니다. (예: 임수정 (배우))
4. 사건/단체는 공식 명칭을 사용합니다. (예: 2022년 FIFA 월드컵)

### 출력 형식:
{{
  "query": "검색할 문서 제목"
}}

### 예시:
- 질문: "임수정 배우의 나이는?" -> {{"query": "임수정 (배우)"}}
- 질문: "카카오뱅크 26주 적금 금리가 뭐야?" -> {{"query": "카카오뱅크"}}
- 질문: "2022년 월드컵 우승국은?" -> {{"query": "2022년 FIFA 월드컵"}}
"""

USER_PROMPT_KEYWORD = """
### [입력]
QUESTION: {QUESTION}
"""

In [13]:
wiki = WikipediaRetriever(
    cfg=WikiConfig(
        lang="ko",
        top_k=8,
        max_length=20000,
        user_agent="MyApp/1.0 (sucruba7@email.com)",
    )
)

In [14]:
keyword = extract_keyword(
    question=q,
    today=TODAY,
    system_prompt=SYSTEM_PROMPT_KEYWORD,
    user_prompt=USER_PROMPT_KEYWORD,
    llm_call=llm_call
)

In [17]:
keyword

KeywordResult(query='2023 APEC 정상회의')

In [15]:
wiki_result = wiki.get_wiki_text(keyword.query)

In [16]:
wiki_result

WikiResult(title='2025년 경주 APEC 정상회의', text='제33회 아시아 태평양 경제협력체 정상회의(영어: APEC 2025 Korea)는 2025년 10월 31일부터 11월 1일까지 대한민국 경상북도 경주시에서 개최된 아시아 태평양 경제협력체(APEC) 연례 회의이다. 2005년 부산 APEC 정상회의 다음으로 대한민국에서 개최된 두 번째 APEC 정상회의이다.\nAPEC 연례 정상회의는 그 해 의장국에서 개최되므로, 2025년 의장국을 맡게 된 대한민국이 개최지로 선정되었다. 비공식고위관리회의를 시작으로 2025년 연중 200회 이상의 각급 APEC 회의가 개최되며 21개국 정상을 비롯하여 6,000여명의 관료와 기업인, 언론인 등이 방한했다.\n\n유치 과정\n2015년 11월, 제23차 APEC 정상회의 참석차 필리핀 마닐라를 방문 중이던 박근혜 대한민국 대통령이 2025년 APEC 정상회의의 대한민국 개최를 제안하였고, 이에 참가국들이 수락함에 따라 2025년 APEC 정상회의의 대한민국 개최가 확정되었다.\n2024년 3월, 외교부는 개최 도시 선정위원회를 구성하여 본격적으로 개최 도시를 선정하기 위한 절차에 돌입하였다. 위원회는 윤진식 한국무역협회 회장을 위원장으로 하였으며, 정부 관료 및 민간위원 등 17명으로 구성되었다. 통과하여야 하는 서면 심사 항목으로는 \'국제회의 개최 시 필요한 객관적 기준 및 역량을 평가한 4개 항목\'과 기타 \'18개 평가지표\'가 있다.\n5월 7일, 2025 APEC 정상회의 개최도시선정위원회 2차 회의가 열렸다. 회의에서는 3월 말부터 4월 19일까지 접수된 각 지자체의 유치 신청서를 검토하였으며 경상북도 경주시와 인천광역시, 제주특별자치도가 후보 도시로 최종 선정되었다. 같은 달 20일부터 22일까지, 후보지 3곳에 대한 현장 실사가 진행되었다. 김철원 경희대학교 교수가 실사단 단장을 맡았으며, 실시단은 각 후보 도시의 회의 시설 및 숙소, 공항 시설 및 교통, 경호 및 안전 여건 등

In [18]:
SYSTEM_PROMPT_RAG_ANSWER = """
당신은 위키피디아 문서를 분석하여 질문에 답하는 전문가입니다.
제공된 위키피디아 텍스트에서 질문 관련 정보를 찾아 답변하세요.

### [기준 정보]
- TODAY: {TODAY}

### [답변 원칙]
1. **위키 우선:** 제공된 위키 텍스트에 답이 있으면 그것을 사용
2. **지식 사용 제한:** 위키 텍스트가 없거나 답이 명시되지 않으면, 확실히 아는 범위에서만 답변 (불확실하면 "정보 없음")
3. **계산 수행:** 날짜/나이/기간은 TODAY 기준 계산
4. **불필요한 부연 금지:** 질문에 대한 짧은 정답(숫자/명사구/한 문장)만 출력

### [중요]
- 위키 텍스트 길어도 끝까지 스캔
- 환각 금지: 모르면 "정보 없음"
- 부연 설명 금지

### [JSON 출력]
{{
  "answer": "답변"
}}

### [예시]

**위키:** "대한민국(1948년 8월 15일 정부 수립)..."
**질문:** "대한민국 건국일?"
**출력:** {{"answer": "1948년 8월 15일"}}

**위키:** "파이썬은 1991년 귀도 반 로섬이 개발..."
**질문:** "파이썬 개발자?"
**출력:** {{"answer": "귀도 반 로섬"}}

**위키:** (없음)
**질문:** "지구 둘레는?"
**출력:** {{"answer": "약 4만 km"}}

**위키:** (나이 정보 없음)
**질문:** "BTS RM 나이?"
**출력:** {{"answer": "정보 없음"}}
"""

USER_PROMPT_RAG_ANSWER = """
### [위키피디아 전문]
{CONTENT}

### [질문]
{QUESTION}

위키에 답이 있으면 사용하고, 없거나 불완전하면 당신의 지식으로 답변하세요.
"""

In [4]:
### 예시용

SAMPLE_N = 50
RANDOM_STATE = 42
OUT_PATH = "./sample_50.csv"   # 원하는 경로/파일명으로 바꿔도 됨

n = min(SAMPLE_N, len(df))
sample_df = df.sample(n=n, random_state=RANDOM_STATE).reset_index(drop=True)

os.makedirs(os.path.dirname(OUT_PATH) or ".", exist_ok=True)
sample_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved {len(sample_df)} rows -> {OUT_PATH}")
sample_df.head(3)

Saved 50 rows -> ./sample_50.csv


,id,split,domain,question,effective_year,next_review,false_premise,num_hops,fact_type,source,answer_0,answer_1,answer_2,answer_3,answer_4,answer_5,answer_6,answer_7,answer_8,answer_9
0,3196,DEV,IT/과학,처녀자리는 어떤 별자리 사이에 놓여 있어?,before 2022,NaN,False,one-hop,never-changing,https://ko.wikipedia.org/wiki/%EC%B2%98%EB%85%...,사자자리와 천칭자리,처녀자리는 사자자리와 천칭자리 사이에 놓여있습니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3080,DEV,경제,2024년 산업은행이 부산에 남부권투자금융본부와 함께 신설하는 조직은 무엇인가요?,2024,NaN,False,one-hop,never-changing,https://n.news.naver.com/mnews/article/658/000...,지역기업종합지원센터,지역기업종합지원센터를 남부권투자금융본부와 함께 신설합니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3481,DEV,경제,상속세 과세표준 10억 이하의 세율은 몇 퍼센트야?,2025,occasionally,False,one-hop,slow-changing,https://easylaw.go.kr/CSP/CnpClsMain.laf?popMe...,30프로입니다.,상속세 과세표준 10억 이하의 세율은 30프로입니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
